In [1]:
def draw_palm_detection(image, box, keypoints, conf):
    """
    image:     numpy array (H, W, 3) - the original input image (any size)
    box:       dict with x_center, y_center, width, height, x_min, y_min, x_max, y_max
               all in 192x192 pixel space
    keypoints: dict of name -> (x, y) in 192x192 pixel space
    conf:      float confidence score
    """
    h, w = image.shape[:2]

    # Scale factor from 192x192 detection space back to original image size
    sx = w / 192.0
    sy = h / 192.0

    def scale(x, y):
        return (int(x * sx), int(y * sy))

    vis = image.copy()

    # --- Bounding box ---
    x_min, y_min = scale(box['x_min'], box['y_min'])
    x_max, y_max = scale(box['x_max'], box['y_max'])
    cv2.rectangle(vis, (x_min, y_min), (x_max, y_max), color=(0, 255, 0), thickness=2)

    # --- Confidence label ---
    label = f"palm {conf:.2f}"
    cv2.putText(vis, label, (x_min, y_min - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # --- Keypoints ---
    colors = {
        'wrist':      (255,   0,   0),
        'pinky_mcp':  (255, 128,   0),
        'index_mcp':  (255, 255,   0),
        'middle_tip': (  0, 255,   0),
        'index_tip':  (  0, 255, 255),
        'thumb_tip':  (  0,   0, 255),
        'thumb_mcp':  (128,   0, 255),
    }
    for name, (kx, ky) in keypoints.items():
        pt = scale(kx, ky)
        cv2.circle(vis, pt, radius=4, color=colors[name], thickness=-1)
        cv2.putText(vis, name, (pt[0] + 5, pt[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, colors[name], 1)

    # --- Skeleton: connect wrist to MCP joints ---
    connections = [
        ('wrist', 'pinky_mcp'),
        ('wrist', 'index_mcp'),
        ('wrist', 'thumb_mcp'),
        # ('index_mcp', 'pinky_mcp'),
    ]
    for a, b in connections:
        pt_a = scale(*keypoints[a])
        pt_b = scale(*keypoints[b])
        cv2.line(vis, pt_a, pt_b, color=(200, 200, 200), thickness=1)

    return vis

import numpy as np
import cv2

def detect_display(path):
    from PIL import Image
    import airlib

    image = Image.open(path).resize((192, 192))
    best_conf, box, keypoints = airlib.detect(image)

    vis = draw_palm_detection(np.array(image, dtype=np.uint8), box, keypoints, float(best_conf))
    from IPython.display import display
    from PIL import Image
    
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(vis))

In [3]:
import numpy as np
import cv2

# ── Landmark connections (finger skeletons) ──────────────────────────────────
HAND_CONNECTIONS = [
    # Thumb
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky
    (0, 17), (17, 18), (18, 19), (19, 20),
    # Palm
    (5, 9), (9, 13), (13, 17),
]

LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]


def draw_landmarks(image: np.ndarray, landmarks: list, draw_labels: bool = True):
    """
    Draw hand skeleton and landmark points over image (in-place).
    """
    img = image.copy()
    h, w = img.shape[:2]

    # Draw connections
    for (a, b) in HAND_CONNECTIONS:
        x1, y1, _ = landmarks[a]
        x1 = int(x1)
        y1 = int(y1)
        x2, y2, _ = landmarks[b]
        x2 = int(x2)
        y2 = int(y2)
        cv2.line(img, (x1, y1), (x2, y2), color=(0, 200, 0), thickness=2)

    # Draw landmark dots and labels
    for i, (px, py, z) in enumerate(landmarks):
        px = int(px)
        py = int(py)
        # Fingertips (indices 4,8,12,16,20) get a larger dot
        is_tip = i in (4, 8, 12, 16, 20)
        radius = 6 if is_tip else 4
        color = (0, 0, 255) if is_tip else (255, 0, 0)
        cv2.circle(img, (px, py), radius, color, -1)
        cv2.circle(img, (px, py), radius, (255, 255, 255), 1)  # white outline

        if draw_labels:
            label = f"{i}"  # just the index; swap for LANDMARK_NAMES[i] if you want full names
            cv2.putText(img, label, (px + 6, py - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

    return img

In [15]:
def landmark_display(path):
    from PIL import Image
    import airlib

    img = Image.open(path)
    
    landmarks, hand_presence, handedness, world_landmarks = airlib.landmark(img)
    vis = draw_landmarks(np.array(img.resize((244, 244)), dtype=np.uint8), airlib.parse_landmarks(landmarks))

    display(Image.fromarray(vis))